In [ ]:
import pandas as pd

# File paths - using the exact ones you provided
mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"          # r"" prevents backslash issues
indent_file  = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# Load files
print("Loading mapping file...")
df_mapping = pd.read_excel(mapping_file)

print("Loading indent file (Sheet1)...")
df_indent = pd.read_excel(indent_file, sheet_name="Sheet1")

# Quick diagnostic prints - run this to confirm columns
print("\nMapping file columns:", df_mapping.columns.tolist())
print("\nIndent file columns:", df_indent.columns.tolist())

# Rename columns (using your latest description)
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',         # child part
    'Sub_Label': 'Switch_Part',         # switch part
    'Main_Count': 'Historical_Main',    # ignored
    'Sub_Count': 'Qty_per_Switch'       # qty of child per switch → this is the key multiplier
})

df_indent = df_indent.rename(columns={
    'Part number': 'Switch_Part'        # matches Sub_Label
})

# Month columns (exact as you said)
month_cols = ["Feb'26", "Mar'26", "Apr'26", "May'26", "Jun'26", "Jul'26"]

# Create clean column names for calculated fields
clean_months = [m.replace("'", "") for m in month_cols]   # Feb'26 → Feb26

# Merge the two tables
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

print(f"\nMerge complete — {len(df_merged)} rows")
if month_cols:
    missing_count = df_merged[month_cols[0]].isna().sum()
    print(f"Rows missing data for {month_cols[0]}: {missing_count}")
    if missing_count > 0:
        print("→ If missing count is high → check if Switch_Part names match exactly between files")

# Calculate daily demand and 2-days requirement
for month, clean in zip(month_cols, clean_months):
    daily_col   = f"Daily_{clean}"
    twodays_col = f"2Days_{clean}"
    
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ─────────────────────────────────────────────
# 1. Totals per child part
# ─────────────────────────────────────────────
totals = df_merged.groupby('Child_Part', as_index=False).agg({
    f"Daily_{clean}": 'sum' for clean in clean_months
})

for clean in clean_months:
    totals[f"2Days_{clean}"] = (totals[f"Daily_{clean}"] * 2).round(2)

totals = totals[
    ['Child_Part'] +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
]

totals.to_excel("Child_Totals_2Days_Per_Month.xlsx", index=False)
print(f"Totals saved: Child_Totals_2Days_Per_Month.xlsx ({len(totals)} rows)")

# ─────────────────────────────────────────────
# 2. Detailed breakdown
# ─────────────────────────────────────────────
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

detailed = df_merged[detailed_cols]
detailed.to_excel("Child_Detailed_Breakdown_2Days.xlsx", index=False)
print(f"Detailed saved: Child_Detailed_Breakdown_2Days.xlsx ({len(detailed)} rows)")

print("\nAll done! Check the two new files in the same folder as the script.")